# NB16 — Final test run on QuranMB.v2 (SPEC T8, `FINAL_RUN=1`)

**The only notebook allowed to open the test set, and it is run once.** Everything is frozen in
`final_config.json` (T7, committed before this run): checkpoints, verifiers, τ values, the LM weight and the
headline system (decision 37: row 3 + deletion rule). Order of operations (hard rule 1):

1. load the test **canonical input only** (`ID`, `Reference_phn`) and audio;
2. run each model once (row 0 = NB8 final, BASE, the 6 ensemble members, stage-1) and cache the logits;
3. build and **write every prediction CSV**;
4. only then load the full truth file (with `Annotation_phn`) and hand it to the official scorer, once per row.

Before touching the test set it re-checks that every verifier reproduces its dev F1, that the config matches
the attached artefacts, and that eager attention matches the default on the GPU. **Second run:** the first run
crashed on the GPU while caching test logits, before the answer key was loaded or anything was scored; inference is
now crash-isolated with eager attention (cell 5). Row 0 should reproduce NB5's 0.667: a check of the whole test pipeline.

**Run settings:** Accelerator **GPU T4 x2**, Internet *On*. Secrets: `HF_TOKEN` (needs access to IqraEval_Test_GT).
Add data: `iqra-cache`, NB11 (`notebook7ed1966e71`), NB8 model (`notebook17aed3d165`), stage-1 model
(`quran-phoneme-model`), `iqra-stage2-run-a`, `notebook7abb4f4be3` (B), `iqra-stage2-base`, NB14-T6
(`notebook0b7d1dd5ea`), NB14 stage-1 caches (`notebookbaca481210`), NB15-T6 (`notebook993e14af56`).

**After the run:** Save Version; make a dataset from the output (evidence). Paste back the last cell.


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment, frozen config, models, verifiers (no test data touched yet) -------------------
os.environ["FINAL_RUN"] = "1"            # hard rule 1: only NB16 may open the test set
import glob, json, queue, threading, time
import numpy as np
import pandas as pd
import torch, lightgbm as lgb
from transformers import AutoFeatureExtractor
from iqra import data, ensemble as E, model as M, parity, posteriors, scorer, transfer as X, verifier
from iqra.candidates import candidate_features
from iqra.config import Config, find_cache_root, find_input
from iqra.vocab import Vocab

DEVICES = [f"cuda:{i}" for i in range(min(torch.cuda.device_count(), 2))] or ["cpu"]
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required (IqraEval_Test_GT is gated)") from e
T_START = time.time()
FINAL = json.load(open(f"{REPO_DIR}/final_config.json"))
cfg = Config()
cfg.paths.cache_root = find_cache_root()
vocab = Vocab.load(cfg.paths.vocab)
TOK2ID, ID2TOK, BLANK = vocab.tok2id, vocab.id2tok, vocab.blank_id
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
T6_DIR = find_input("nb14_t6_report.json", what="the NB14-T6 output")
S1_DIR = find_input("post_stage1_extra_dev.pt", what="the NB14 output (stage-1 caches)")
V_DIR = find_input("nb15_t6_report.json", what="the NB15-T6 output")
STAGE1 = find_input("checkpoint-5000/config.json", what="quran-phoneme-model (checkpoint-5000)")
OUT = "/kaggle/working/nb16_final"
os.makedirs(OUT, exist_ok=True)

runs = E.find_runs(["/kaggle/input"])
by_name = {m.name: m for r in runs.values() for m in r["members"]}
members = [m["name"] for m in FINAL["ensemble_members"]]
assert all(n in by_name for n in members + [FINAL["base_checkpoint"]["name"]]), sorted(by_name)
for m in FINAL["ensemble_members"] + [FINAL["base_checkpoint"]]:
    assert abs(by_name[m["name"]].f1 - m["dev_f1"]) < 1e-4, m
t6 = json.load(open(f"{V_DIR}/nb15_t6_report.json"))
assert t6["ladder"]["row4_ens_primary_verifier"]["tau"] == FINAL["row4"]["tau"]
assert t6["ladder"]["row6_stage1_primary_verifier"]["tau"] == FINAL["row6"]["tau"]
assert t6["ladder"]["row5_ens_lm"]["alpha"] == FINAL["row5"]["lm_alpha"] == 0.0

par = parity.verify_environment(DEVICES[0])          # decision 27; also finds the NB8 model by its hash
assert par["model_dir"] and parity.sha256_file(f"{par['model_dir']}/model.safetensors") == FINAL["row0_model"]["weights_sha256"]
MODELS = {"row0_nb8": par["model_dir"], "stage1": STAGE1, FINAL["base_checkpoint"]["name"]: by_name[FINAL["base_checkpoint"]["name"]].path}
MODELS.update({n: by_name[n].path for n in members})
print("version parity: PASS |", par["here"], "| devices:", DEVICES)
for k, v in MODELS.items():
    print(f"  {k:10s} {v}")
fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
report = {"repo_commit": REPO_COMMIT, "versions": par["here"], "devices": DEVICES, "headline": FINAL["headline"],
          "models": MODELS}


In [ ]:
# ---- Verifiers: rebuild, and reproduce every dev F1 before the test set is opened ------------------
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
EXP_DEV = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"].str.split()))

def dev_f1(tok):
    pred = pd.DataFrame({"ID": list(tok), "Prediction": [" ".join(t) for t in tok.values()]})
    return round(scorer.summed_metrics(scorer.per_clip_counts(truth_dev, pred))["f1"], 4)

def tables(tag):
    tr = pd.concat([pd.read_parquet(f"{V_DIR}/candidates_{tag}_{s}.parquet") for s in ("fold_a", "fold_b", "iqra_dev_slice")],
                   ignore_index=True)
    return tr, pd.read_parquet(f"{V_DIR}/candidates_{tag}_extra_dev.parquet")

def primary(tag, model_file):
    tr, dv = tables(tag)
    enc = verifier.Encoder(verifier.FEATURE_SETS["primary"]).fit(tr[tr["op"].isin(verifier.PRIMARY_OPS)].reset_index(drop=True))
    return lgb.Booster(model_file=f"{V_DIR}/{model_file}"), enc, dv

dev_caches = {}
for p in sorted(glob.glob(f"{T6_DIR}/post_*_extra_dev.pt")):
    meta, clips = posteriors.read_cache(p)
    dev_caches[meta["checkpoint"]] = clips
dev_ens = E.ensemble_split({n: dev_caches[n] for n in members}, members, ID2TOK, BLANK)
HEARD_DEV = {c: e["tokens"] for c, e in dev_ens.items()}
HEARD_DEV_S1 = {c: e["tokens"] for c, e in posteriors.read_cache(f"{S1_DIR}/post_stage1_extra_dev.pt")[1].items()}

b4, enc4, dv4 = primary("ens", FINAL["row4"]["verifier"])
f4 = dev_f1(verifier.apply(dv4, verifier.predict_primary(b4, enc4, dv4), EXP_DEV, HEARD_DEV, FINAL["row4"]["tau"], revert_deletions=True))
b6, enc6, dv6 = primary("stage1", FINAL["row6"]["verifier"])
f6 = dev_f1(verifier.apply(dv6, verifier.predict_primary(b6, enc6, dv6), EXP_DEV, HEARD_DEV_S1, FINAL["row6"]["tau"], revert_deletions=True))
tr_all, dv_all = tables("ens")
bF, encF = verifier.train(tr_all, FINAL["secondary_full_verifier"]["n_rounds"], cfg.verifier)
fF = dev_f1(verifier.apply(dv_all, verifier.predict(bF, encF, dv_all), EXP_DEV, HEARD_DEV, FINAL["secondary_full_verifier"]["tau"]))
fD = dev_f1({c: X.deletion_rule(EXP_DEV[c], t) for c, t in HEARD_DEV.items()})
repro = {"row4": (f4, FINAL["row4"]["dev_f1"]), "row6": (f6, FINAL["row6"]["dev_f1"]),
         "secondary_full_verifier": (fF, FINAL["secondary_full_verifier"]["dev_f1"]),
         "row3_deletion_rule": (fD, FINAL["dev_f1"]["row3_deletion_rule"])}
print("dev reproduction (got, frozen):", repro)
assert all(abs(a - b) < 1e-4 for a, b in repro.values()), "a frozen component does not reproduce its dev F1: stop"
report["dev_reproduction"] = repro

# Eager vs default attention on the GPU, on the parity fixture's two real recitations (no test data)
meta_ref, ref_arrays = parity.load_fixture()
real = {k[: -len("_features_input")]: v for k, v in ref_arrays.items() if k.endswith("_features_input")}
fe_ref = AutoFeatureExtractor.from_pretrained(par["model_dir"])
outs = {}
for impl in ("default", "eager"):
    m_, info_ = M.load_model(par["model_dir"], DEVICES[0], attn_implementation=None if impl == "default" else "eager")
    assert not any(info_.values()), info_
    outs[impl] = parity.run(m_, fe_ref, DEVICES[0], real)
    del m_
    torch.cuda.empty_cache()
eager_check = {k: {"max_abs_logit_diff": float(np.max(np.abs(outs["eager"][k] - outs["default"][k]))),
                   "argmax_mismatch_frames": int(np.sum(outs["eager"][k].argmax(-1) != outs["default"][k].argmax(-1))),
                   "vs_4.44.2_argmax_mismatch": int(np.sum(outs["eager"][k].argmax(-1) != ref_arrays[k].argmax(-1)))
                   if outs["eager"][k].shape == ref_arrays[k].shape else None}
               for k in outs["eager"] if k.endswith("_logits") and k in ref_arrays}
print("eager vs default attention (GPU, real recitations):", eager_check)
# same maths in a different kernel: allow at most one near-tied frame per clip to flip
assert all(v["argmax_mismatch_frames"] <= 1 and v["max_abs_logit_diff"] < 1e-2 for v in eager_check.values()), \
    "eager attention does not match the default: stop"
report["eager_vs_default"] = eager_check


In [ ]:
# ---- Test canonical input and audio (Annotation_phn is NOT loaded here) ------------------------------
canon = data.load_test_canonical()                    # ID + Reference_phn only (decision 19)
assert list(canon.columns) == ["ID", "Reference_phn"]
EXP = dict(zip(canon["ID"].astype(str), canon["Reference_phn"].astype(str).str.split()))
test_ds = data.load_test_audio()
ids = [str(x) for x in list(test_ds["ID"])]
AUDIO = {ids[k]: data.to_16k(arr, sr) for k, arr, sr in data.iter_audio(test_ds)}
assert set(AUDIO) == set(EXP), (len(AUDIO), len(EXP))
oov = sorted({t for v in EXP.values() for t in v if t not in TOK2ID})
hours = sum(len(a) for a in AUDIO.values()) / 16_000 / 3600
print(f"test clips {len(AUDIO)} | {hours:.2f} h | canonical tokens outside the vocab: {oov}")
report["test"] = {"clips": len(AUDIO), "hours": round(hours, 3), "oov_tokens": oov}


In [ ]:
# ---- Run each model once on the test set: crash-isolated, eager attention (see the first-run note) -----
# First NB16 run: "CUDA error: illegal memory access" on every model on both GPUs while caching test logits
# (answer key never loaded, nothing scored). transformers 5.x runs w2v-BERT attention with its relative-position
# bias through the fused SDPA kernel; 4.44.2 (NB5's 0.667) used the plain computation. So: eager attention on the
# GPU (checked against the default above), each model in a child process; a clip that still crashes the GPU is
# re-run on CPU with the same checkpoint (counted in the report).
from iqra import safe_infer as SI
from transformers import AutoConfig
AUDIO_NPZ, WORK = "/tmp/nb16_audio.npz", "/tmp/nb16_work"
SI.save_audio(AUDIO, AUDIO_NPZ)
dur = np.array([len(a) for a in AUDIO.values()]) / 16_000
print(f"test clip seconds: min {dur.min():.2f} | median {np.median(dur):.2f} | p99 {np.quantile(dur, .99):.2f} | max {dur.max():.2f}")
report["test"]["seconds"] = {"min": round(float(dur.min()), 2), "median": round(float(np.median(dur)), 2),
                             "max": round(float(dur.max()), 2)}
GPUS = list(range(len(DEVICES))) if DEVICES[0] != "cpu" else [None]
jobs, lock, errors, TEST, timing, runinfo = queue.Queue(), threading.Lock(), [], {}, {}, {}
for name in MODELS:
    jobs.put(name)

def worker(gpu):
    while True:
        try:
            name = jobs.get_nowait()
        except queue.Empty:
            return
        try:
            logits, rep = SI.run_model(name, MODELS[name], AUDIO_NPZ, WORK, gpu_index=gpu, attn="eager",
                                       log=lambda m: print(m, flush=True))
            assert set(logits) == set(AUDIO), name
            entries = {cid: posteriors.make_entry(logits[cid], ID2TOK, BLANK) for cid in AUDIO}
            posteriors.write_cache(f"{OUT}/post_{name}_test.pt", entries, checkpoint=name, split="test",
                                   vocab_hash=vocab.hash, frame_rate=posteriors.model_frame_rate(AutoConfig.from_pretrained(MODELS[name])),
                                   extra_meta={"repo_commit": REPO_COMMIT, "attn": "eager", "cpu_clips": len(rep["cpu_clips"])
                                               if isinstance(rep["cpu_clips"], list) else rep["cpu_clips"]})
            with lock:
                TEST[name], timing[name], runinfo[name] = entries, rep["minutes"], rep
                print(f"[gpu {gpu}] {name}: {rep['minutes']} min | GPU restarts {rep['gpu_restarts']} | "
                      f"CPU clips {len(rep['cpu_clips']) if isinstance(rep['cpu_clips'], list) else rep['cpu_clips']} | "
                      f"{len(TEST)}/{len(MODELS)} done", flush=True)
        except Exception as e:
            with lock:
                errors.append((name, repr(e)[-1500:]))
                print(f"[gpu {gpu}] {name} FAILED: {e!r}"[-1500:], flush=True)

threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
for t in threads:
    t.start()
for t in threads:
    t.join()
assert not errors, errors
report["inference_minutes"] = timing
report["inference_runs"] = {k: {"gpu_restarts": v["gpu_restarts"], "cpu_clips": v["cpu_clips"]} for k, v in runinfo.items()}


In [ ]:
# ---- Build every system and WRITE every prediction CSV (truth still unopened) -------------------------
bad = E.frames_agree({n: TEST[n] for n in members}, members)
assert not bad, bad[:3]
ens = E.ensemble_split({n: TEST[n] for n in members}, members, ID2TOK, BLANK)
tok = lambda entries: {c: e["tokens"] for c, e in entries.items()}

def verify(entries, booster, enc, tau, revert_deletions, use_primary=True):
    rows, skipped = [], []
    for cid, e in entries.items():
        if any(t not in TOK2ID for t in EXP[cid]):
            skipped.append(cid)                       # unseen canonical token: keep the plain decode
            continue
        rows += candidate_features(cid, EXP[cid], e, TOK2ID, ID2TOK, BLANK, said=None)
    df = pd.DataFrame(rows)
    heard = tok(entries)
    if df.empty:
        return heard, skipped
    probs = verifier.predict_primary(booster, enc, df) if use_primary else verifier.predict(booster, enc, df)
    return verifier.apply(df, probs, EXP, heard, tau, revert_deletions=revert_deletions), skipped

SYSTEMS = {"row0_nb8": tok(TEST["row0_nb8"]), "row1_base": tok(TEST[FINAL["base_checkpoint"]["name"]]),
           "row2_best_single": tok(TEST[FINAL["best_single"]]), "row3_ensemble6": tok(ens),
           "row7_stage1": tok(TEST["stage1"])}
SYSTEMS["row5_ens_lm"] = dict(SYSTEMS["row3_ensemble6"])            # frozen LM weight 0: identical to row 3
SYSTEMS["row4_ens_primary_verifier"], sk4 = verify(ens, b4, enc4, FINAL["row4"]["tau"], True)
SYSTEMS["row6_stage1_primary_verifier"], sk6 = verify(TEST["stage1"], b6, enc6, FINAL["row6"]["tau"], True)
SYSTEMS["secondary_full_verifier"], skF = verify(ens, bF, encF, FINAL["secondary_full_verifier"]["tau"], False, use_primary=False)
for src, key in (("row3_ensemble6", "row3_deletion_rule"), ("row0_nb8", "row0_deletion_rule"),
                 ("row2_best_single", "row2_deletion_rule"), ("row7_stage1", "row7_deletion_rule")):
    SYSTEMS[key] = {c: X.deletion_rule(EXP[c], t) for c, t in SYSTEMS[src].items()}
assert sorted(SYSTEMS) == sorted(FINAL["test_rows"]), sorted(set(SYSTEMS) ^ set(FINAL["test_rows"]))
PRED = {}
for key, t in SYSTEMS.items():
    PRED[key] = pd.DataFrame({"ID": list(t), "Prediction": [" ".join(v) for v in t.values()]})
    PRED[key].to_csv(f"{OUT}/pred_{key}_test.csv", index=False)
print("prediction CSVs written:", len(PRED), "| clips skipped by verifiers (unseen tokens):",
      {"row4": len(sk4), "row6": len(sk6), "full": len(skF)})
report["verifier_skipped_clips"] = {"row4": len(sk4), "row6": len(sk6), "full": len(skF)}


In [ ]:
# ---- Only now: the full truth file goes to the official scorer, once per row --------------------------
truth = data.load_test_truth_for_scorer()
truth["ID"] = truth["ID"].astype(str)
assert set(truth["ID"]) == set(EXP)
CL = dict(zip(truth["ID"], truth["Reference_phn"]))          # cluster bootstrap by canonical sentence
res, COUNTS = [], {}
for key in FINAL["test_rows"]:
    r = scorer.official_score(truth, PRED[key], keep_dir=f"{OUT}/score_{key}")
    COUNTS[key] = scorer.per_clip_counts(truth, PRED[key])
    assert int(COUNTS[key]["TR"].sum()) == r["TR"] and int(COUNTS[key]["FR"].sum()) == r["FR"]
    res.append({"row": key, "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "per": r["per"],
                "TR": r["TR"], "FR": r["FR"], "FA": r["FA"], "dev_f1": FINAL["dev_f1"].get(key),
                "projected_test_f1": FINAL["projected_test_f1"].get(key), "headline": key == FINAL["headline"]})
results = pd.DataFrame(res)
results["inference_minutes"] = results["row"].map({"row0_nb8": timing["row0_nb8"], "row1_base": timing[FINAL["base_checkpoint"]["name"]],
                                                    "row2_best_single": timing[FINAL["best_single"]], "row7_stage1": timing["stage1"]})
results.to_csv(f"{OUT}/results.csv", index=False)
with pd.option_context("display.width", 220):
    print(results.drop(columns=["TR", "FR", "FA"]).to_string(index=False))

def gain(new, base):
    g = scorer.paired_bootstrap_gain(COUNTS[base], COUNTS[new], CL)
    return {"gain": round(g["gain"], 4), "ci90": [round(g["lo"], 4), round(g["hi"], 4)], "n_sentences": g["n_clusters"]}

GAINS = {"headline_vs_row0": gain(FINAL["headline"], "row0_nb8"), "row4_vs_row3": gain("row4_ens_primary_verifier", "row3_ensemble6"),
         "row3_deletion_vs_row3": gain("row3_deletion_rule", "row3_ensemble6"), "row3_vs_row1": gain("row3_ensemble6", "row1_base"),
         "row6_vs_row7": gain("row6_stage1_primary_verifier", "row7_stage1")}
for k, v in GAINS.items():
    print(k, v)
h = results[results["headline"]].iloc[0]
r0 = results[results["row"] == "row0_nb8"].iloc[0]
print(f"\nrow 0 reproduces NB5? F1 {r0['f1']} (NB5: {FINAL['row0_model']['test_f1_nb5']})")
print(f">>> HEADLINE ({FINAL['headline']}): F1 {h['f1']}  P {h['precision']}  R {h['recall']}  | "
      f"IQRA 2026 best {FINAL['target']['iqra2026_best']}: {'BEATEN' if h['f1'] > FINAL['target']['iqra2026_best'] else 'not beaten'}")
report.update({"results": res, "gains": GAINS, "minutes": round((time.time() - T_START) / 60, 1)})
with open(f"{OUT}/nb16_report.json", "w") as f:
    json.dump(report, f, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------------------------
print(json.dumps(report, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
